# Nano GTSRB FINN — consolidated demo

Same pipeline as `demo.py`: camera thread, FINN batch=1, temporal sum over the latest 10 frames, and full-frame preview with the inference crop marked by a green rectangle.

In [ ]:
from pathlib import Path
import sys

PROJECT_DIR = Path('/home/xilinx/nano_gtsrb_demo')

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from config import DemoConfig
from pipe_manager import PipeManager

config = DemoConfig(
    target_fps=20.0,
    crop_fraction=0.30,
    tilt_fraction=0.20,
    score_window=10,
)

manager = PipeManager(
    config,
    clear_pynq_state=True,
)

print('Project:', PROJECT_DIR)
print('Python:', sys.executable)


In [ ]:
import cv2
from IPython.display import display, Image
from visualization import annotate_frame

DISPLAY_EVERY_N_FRAMES = 4  # ~5 FPS preview at 20 inference FPS
display_handle = None
display_counter = 0

def notebook_update(snapshot, frame_bgr):
    global display_handle, display_counter

    display_counter += 1

    if display_counter % DISPLAY_EVERY_N_FRAMES != 0:
        return

    preview = annotate_frame(
        frame_bgr,
        snapshot,
        config.crop_fraction,
        config.tilt_fraction,
        config.input_size,
    )

    ok, jpg = cv2.imencode(
        '.jpg',
        preview,
        [cv2.IMWRITE_JPEG_QUALITY, 75],
    )

    if not ok:
        return

    image = Image(data=jpg.tobytes())

    if display_handle is None:
        display_handle = display(image, display_id=True)
    else:
        display_handle.update(image)


In [ ]:
# Interrupt the cell / Ctrl+C to stop cleanly.
manager.run(on_update=notebook_update)
